
# Project 23: Agentic AI for Intelligent Business Process Automation

## Notebook 02 — Data Preprocessing & Feature Engineering

**Dataset:** Walmart M5 Forecasting Dataset

### Objectives
1. Prepare historical sales data for forecasting.
2. Transform daily sales from wide to long format.
3. Integrate calendar and pricing information.
4. Handle missing values appropriately.
5. Engineer time-series forecasting features.
6. Prepare chronological training, validation, and test datasets.



In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

calendar = pd.read_csv("C:\\Projects\\23_Agentic_Business_AI\\data\\calendar.csv")
prices = pd.read_csv("C:\\Projects\\23_Agentic_Business_AI\\data\\sell_prices.csv")
sales = pd.read_csv("C:\\Projects\\23_Agentic_Business_AI\\data\\sales_train_validation.csv")

print("Calendar:", calendar.shape)
print("Prices:", prices.shape)
print("Sales:", sales.shape)


Calendar: (1969, 14)
Prices: (6841121, 4)
Sales: (30490, 1919)


## 2. Data Selection
### 2.1 Filter Store CA_1

In [2]:

# Select the initial store
store_id = "CA_1"

# Filter sales and prices
sales_ca1 = sales[
    sales["store_id"] == store_id
].copy()

prices_ca1 = prices[
    prices["store_id"] == store_id
].copy()

# Check dimensions
print("Sales CA_1:", sales_ca1.shape)
print("Prices CA_1:", prices_ca1.shape)

# Verify product categories
print("\nCategories:")
print(sales_ca1["cat_id"].value_counts())

# Verify unique products
print("\nUnique products:", sales_ca1["item_id"].nunique())


Sales CA_1: (3049, 1919)
Prices CA_1: (698412, 4)

Categories:
cat_id
FOODS        1437
HOUSEHOLD    1047
HOBBIES       565
Name: count, dtype: int64

Unique products: 3049


### 2.2 Reshape Sales Data from Wide to Long Format

In [3]:

# Identify daily sales columns
day_cols = sales_ca1.filter(regex=r"^d_\d+$").columns

# Identify product information columns
id_cols = [
    "id", "item_id", "dept_id",
    "cat_id", "store_id", "state_id"
]

# Convert wide format to long format
sales_long = sales_ca1.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

# Reduce memory usage
sales_long["sales"] = sales_long["sales"].astype("int32")

# Display results
print("Long-format shape:", sales_long.shape)
print("\nSample records:")
print(sales_long.head())


Long-format shape: (5832737, 8)

Sample records:
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_004_CA_1_validation  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_005_CA_1_validation  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1   

  state_id    d  sales  
0       CA  d_1      0  
1       CA  d_1      0  
2       CA  d_1      0  
3       CA  d_1      0  
4       CA  d_1      0  


### 2.3 Merge Sales with Calendar Features

In [4]:

# Prepare calendar data
calendar["date"] = pd.to_datetime(calendar["date"])

# Select relevant calendar features
calendar_cols = [
    "d",
    "date",
    "wm_yr_wk",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA"
]

# Merge daily sales with calendar information
sales_calendar = sales_long.merge(
    calendar[calendar_cols],
    on="d",
    how="left",
    validate="many_to_one"
)

# Verify the merge
print("Merged shape:", sales_calendar.shape)

print(
    "Missing dates:",
    sales_calendar["date"].isna().sum()
)

print(
    "Missing week identifiers:",
    sales_calendar["wm_yr_wk"].isna().sum()
)

# Display sample records
print("\nSample records:")
print(
    sales_calendar[
        ["item_id", "date", "sales", "wday",
         "event_name_1", "snap_CA"]
    ].head()
)


Merged shape: (5832737, 18)
Missing dates: 0
Missing week identifiers: 0

Sample records:
         item_id       date  sales  wday event_name_1  snap_CA
0  HOBBIES_1_001 2011-01-29      0     1          NaN        0
1  HOBBIES_1_002 2011-01-29      0     1          NaN        0
2  HOBBIES_1_003 2011-01-29      0     1          NaN        0
3  HOBBIES_1_004 2011-01-29      0     1          NaN        0
4  HOBBIES_1_005 2011-01-29      0     1          NaN        0


### 2.4 Merge Weekly Selling Prices

In [5]:

# Merge sales-calendar data with weekly selling prices
full_data = sales_calendar.merge(
    prices_ca1,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

# Verify merge results
print("Final merged shape:", full_data.shape)

print(
    "Missing selling prices:",
    full_data["sell_price"].isna().sum()
)

print(
    "Missing price percentage:",
    f'{full_data["sell_price"].isna().mean() * 100:.2f}%'
)

print("\nSample records:")
print(
    full_data[
        ["item_id", "date", "sales",
         "sell_price", "wm_yr_wk"]
    ].head()
)


Final merged shape: (5832737, 19)
Missing selling prices: 1129842
Missing price percentage: 19.37%

Sample records:
         item_id       date  sales  sell_price  wm_yr_wk
0  HOBBIES_1_001 2011-01-29      0         NaN     11101
1  HOBBIES_1_002 2011-01-29      0         NaN     11101
2  HOBBIES_1_003 2011-01-29      0         NaN     11101
3  HOBBIES_1_004 2011-01-29      0         NaN     11101
4  HOBBIES_1_005 2011-01-29      0         NaN     11101


### 2.5 Investigating Missing Selling Prices

In [6]:

# Identify records with missing selling prices
missing_price_mask = full_data["sell_price"].isna()

missing_price_data = full_data.loc[missing_price_mask]

# Examine sales activity
zero_sales_count = (
    missing_price_data["sales"] == 0
).sum()

positive_sales_count = (
    missing_price_data["sales"] > 0
).sum()

print("Total missing price records:",
      len(missing_price_data))

print("Missing price + zero sales:",
      zero_sales_count)

print("Missing price + positive sales:",
      positive_sales_count)

print("\nPercentage with zero sales:",
      f"{zero_sales_count / len(missing_price_data) * 100:.2f}%")


Total missing price records: 1129842
Missing price + zero sales: 1129842
Missing price + positive sales: 0

Percentage with zero sales: 100.00%


### 2.6 Missing Price Timing Analysis

In [7]:

# Create missing-price indicator
full_data["price_missing"] = (
    full_data["sell_price"].isna()
).astype("int8")

# First and last dates with recorded prices for each product
price_dates = (
    full_data.loc[
        full_data["sell_price"].notna(),
        ["item_id", "date"]
    ]
    .groupby("item_id")["date"]
    .agg(first_price_date="min", last_price_date="max")
)

# Analyze only rows with missing prices
missing = full_data.loc[
    full_data["price_missing"] == 1,
    ["item_id", "date"]
].join(price_dates, on="item_id")

before_first = (
    missing["date"] < missing["first_price_date"]
).sum()

after_last = (
    missing["date"] > missing["last_price_date"]
).sum()

between_prices = (
    (missing["date"] >= missing["first_price_date"]) &
    (missing["date"] <= missing["last_price_date"])
).sum()

print(f"Before first recorded price: {before_first:,}")
print(f"After last recorded price: {after_last:,}")
print(f"Between recorded price dates: {between_prices:,}")

print("\nTotal classified:",
      f"{before_first + after_last + between_prices:,}")


Before first recorded price: 1,129,842
After last recorded price: 0
Between recorded price dates: 0

Total classified: 1,129,842


### 2.7 Filter Pre-Introduction Observations

In [8]:

# Preserve full dataset and filter to recorded-price periods
model_data = full_data.loc[
    full_data["sell_price"].notna()
].copy()

# Verify filtering
print("Original observations:", f"{len(full_data):,}")
print("Filtered observations:", f"{len(model_data):,}")
print("Removed observations:",
      f"{len(full_data) - len(model_data):,}")

print("\nRemaining missing prices:",
      model_data["sell_price"].isna().sum())

print("Remaining unique products:",
      model_data["item_id"].nunique())

print("Remaining stores:",
      model_data["store_id"].unique())


Original observations: 5,832,737
Filtered observations: 4,702,895
Removed observations: 1,129,842

Remaining missing prices: 0
Remaining unique products: 3049
Remaining stores: ['CA_1']


### 2.8 Sort Time-Series Observations

In [9]:

# Sort observations by product and date
model_data = (
    model_data
    .sort_values(["item_id", "date"])
    .reset_index(drop=True)
)

# Check chronological ordering
is_sorted = (
    model_data.groupby("item_id")["date"]
    .apply(lambda x: x.is_monotonic_increasing)
    .all()
)

# Check duplicate product-date combinations
duplicate_dates = model_data.duplicated(
    subset=["item_id", "date"]
).sum()

print("Chronologically sorted:", is_sorted)
print("Duplicate product-date records:", duplicate_dates)

print("\nSample records:")
print(
    model_data[
        ["item_id", "date", "sales", "sell_price"]
    ].head()
)


Chronologically sorted: True
Duplicate product-date records: 0

Sample records:
       item_id       date  sales  sell_price
0  FOODS_1_001 2011-01-29      3         2.0
1  FOODS_1_001 2011-01-30      0         2.0
2  FOODS_1_001 2011-01-31      0         2.0
3  FOODS_1_001 2011-02-01      1         2.0
4  FOODS_1_001 2011-02-02      4         2.0


### 2.9 Verify Daily Time-Series Continuity

In [10]:

# Calculate the gap between consecutive dates
date_gaps = (
    model_data.groupby("item_id")["date"]
    .diff()
)

# Identify gaps larger than one day
gap_mask = date_gaps > pd.Timedelta(days=1)

print("Number of date gaps:", gap_mask.sum())

print(
    "Products with date gaps:",
    model_data.loc[gap_mask, "item_id"].nunique()
)

print(
    "Maximum gap:",
    date_gaps.max()
)

# Show examples if gaps exist
if gap_mask.any():
    print("\nExample records with gaps:")
    print(
        model_data.loc[
            gap_mask,
            ["item_id", "date", "sales"]
        ].head(10)
    )


Number of date gaps: 0
Products with date gaps: 0
Maximum gap: 1 days 00:00:00


### 2.10 Historical Demand Lag Features

In [11]:

# Create historical demand features
# Every row is already sorted by product and date.

grouped_sales = model_data.groupby("item_id")["sales"]

model_data["lag_28"] = grouped_sales.shift(28)
model_data["lag_35"] = grouped_sales.shift(35)

# Verify lag feature generation
print(
    model_data[
        ["item_id", "date", "sales", "lag_28", "lag_35"]
    ].head(40)
)

print("\nMissing lag_28:",
      model_data["lag_28"].isna().sum())

print("Missing lag_35:",
      model_data["lag_35"].isna().sum())


        item_id       date  sales  lag_28  lag_35
0   FOODS_1_001 2011-01-29      3     NaN     NaN
1   FOODS_1_001 2011-01-30      0     NaN     NaN
2   FOODS_1_001 2011-01-31      0     NaN     NaN
3   FOODS_1_001 2011-02-01      1     NaN     NaN
4   FOODS_1_001 2011-02-02      4     NaN     NaN
5   FOODS_1_001 2011-02-03      2     NaN     NaN
6   FOODS_1_001 2011-02-04      0     NaN     NaN
7   FOODS_1_001 2011-02-05      2     NaN     NaN
8   FOODS_1_001 2011-02-06      0     NaN     NaN
9   FOODS_1_001 2011-02-07      0     NaN     NaN
10  FOODS_1_001 2011-02-08      0     NaN     NaN
11  FOODS_1_001 2011-02-09      0     NaN     NaN
12  FOODS_1_001 2011-02-10      3     NaN     NaN
13  FOODS_1_001 2011-02-11      1     NaN     NaN
14  FOODS_1_001 2011-02-12      3     NaN     NaN
15  FOODS_1_001 2011-02-13      0     NaN     NaN
16  FOODS_1_001 2011-02-14      2     NaN     NaN
17  FOODS_1_001 2011-02-15      1     NaN     NaN
18  FOODS_1_001 2011-02-16      2     NaN     NaN


### 2.11 Historical Rolling Demand Features

In [12]:

# Historical sales available for a 28-day forecast
shifted_sales = (
    model_data.groupby("item_id")["sales"]
    .shift(28)
)

# Rolling demand features
model_data["rolling_mean_7"] = (
    shifted_sales
    .groupby(model_data["item_id"])
    .transform(
        lambda x: x.rolling(window=7, min_periods=7).mean()
    )
)

model_data["rolling_mean_28"] = (
    shifted_sales
    .groupby(model_data["item_id"])
    .transform(
        lambda x: x.rolling(window=28, min_periods=28).mean()
    )
)

# Check generated features
print(
    model_data[
        [
            "item_id",
            "date",
            "sales",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_28"
        ]
    ].iloc[25:65]
)

print("\nMissing rolling_mean_7:",
      model_data["rolling_mean_7"].isna().sum())

print("Missing rolling_mean_28:",
      model_data["rolling_mean_28"].isna().sum())


        item_id       date  sales  lag_28  rolling_mean_7  rolling_mean_28
25  FOODS_1_001 2011-02-23      2     NaN             NaN              NaN
26  FOODS_1_001 2011-02-24      2     NaN             NaN              NaN
27  FOODS_1_001 2011-02-25      4     NaN             NaN              NaN
28  FOODS_1_001 2011-02-26      2     3.0             NaN              NaN
29  FOODS_1_001 2011-02-27      2     0.0             NaN              NaN
30  FOODS_1_001 2011-02-28      0     0.0             NaN              NaN
31  FOODS_1_001 2011-03-01      2     1.0             NaN              NaN
32  FOODS_1_001 2011-03-02      1     4.0             NaN              NaN
33  FOODS_1_001 2011-03-03      7     2.0             NaN              NaN
34  FOODS_1_001 2011-03-04      1     0.0        1.428571              NaN
35  FOODS_1_001 2011-03-05      2     2.0        1.285714              NaN
36  FOODS_1_001 2011-03-06      3     0.0        1.285714              NaN
37  FOODS_1_001 2011-03-0

### 2.12 Calendar-Based Feature Engineering

In [13]:

# Extract calendar features
model_data["day_of_week"] = model_data["date"].dt.dayofweek

model_data["is_weekend"] = (
    model_data["day_of_week"] >= 5
).astype("int8")

model_data["day_of_month"] = (
    model_data["date"].dt.day.astype("int8")
)

model_data["quarter"] = (
    model_data["date"].dt.quarter.astype("int8")
)

# Event indicators
model_data["is_event"] = (
    model_data["event_name_1"].notna() |
    model_data["event_name_2"].notna()
).astype("int8")

model_data["is_christmas"] = (
    (model_data["date"].dt.month == 12) &
    (model_data["date"].dt.day == 25)
).astype("int8")

# Inspect the results
print(
    model_data[
        [
            "date",
            "day_of_week",
            "is_weekend",
            "day_of_month",
            "quarter",
            "is_event",
            "is_christmas"
        ]
    ].head(10)
)

print("\nMissing calendar features:")
print(
    model_data[
        [
            "day_of_week",
            "is_weekend",
            "day_of_month",
            "quarter",
            "is_event",
            "is_christmas"
        ]
    ].isna().sum()
)


        date  day_of_week  is_weekend  day_of_month  quarter  is_event  \
0 2011-01-29            5           1            29        1         0   
1 2011-01-30            6           1            30        1         0   
2 2011-01-31            0           0            31        1         0   
3 2011-02-01            1           0             1        1         0   
4 2011-02-02            2           0             2        1         0   
5 2011-02-03            3           0             3        1         0   
6 2011-02-04            4           0             4        1         0   
7 2011-02-05            5           1             5        1         0   
8 2011-02-06            6           1             6        1         1   
9 2011-02-07            0           0             7        1         0   

   is_christmas  
0             0  
1             0  
2             0  
3             0  
4             0  
5             0  
6             0  
7             0  
8             0  
9    

### 2.13 Chronological Train-Validation-Test Split

In [14]:

# Determine the available sales period
min_date = model_data["date"].min()
max_date = model_data["date"].max()

# Reserve two consecutive 28-day periods
forecast_horizon = 28

test_start = max_date - pd.Timedelta(days=27)
val_start = test_start - pd.Timedelta(days=28)

train_end = val_start - pd.Timedelta(days=1)
val_end = test_start - pd.Timedelta(days=1)

print("Data start:", min_date.date())
print("Data end:", max_date.date())

print("\nTraining ends:", train_end.date())
print("Validation:", val_start.date(), "to", val_end.date())
print("Test:", test_start.date(), "to", max_date.date())

print("\nValidation days:",
      (val_end - val_start).days + 1)
print("Test days:",
      (max_date - test_start).days + 1)


Data start: 2011-01-29
Data end: 2016-04-24

Training ends: 2016-02-28
Validation: 2016-02-29 to 2016-03-27
Test: 2016-03-28 to 2016-04-24

Validation days: 28
Test days: 28


### 2.14 Create Chronological Training, Validation, and Test Sets

In [15]:

# Split observations chronologically
train_data = model_data.loc[
    model_data["date"] <= train_end
].copy()

val_data = model_data.loc[
    (model_data["date"] >= val_start) &
    (model_data["date"] <= val_end)
].copy()

test_data = model_data.loc[
    model_data["date"] >= test_start
].copy()

# Verify the splits
print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("Test shape:", test_data.shape)

print("\nDate ranges:")
for name, df in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    print(
        f"{name}: {df['date'].min().date()} "
        f"to {df['date'].max().date()}"
    )

# Ensure the splits do not overlap
assert train_data["date"].max() < val_data["date"].min()
assert val_data["date"].max() < test_data["date"].min()

# Verify no observations were lost
assert (
    len(train_data) + len(val_data) + len(test_data)
    == len(model_data)
)

print("\nChronological split validation: PASSED")


Training shape: (4532151, 30)
Validation shape: (85372, 30)
Test shape: (85372, 30)

Date ranges:
Train: 2011-01-29 to 2016-02-28
Validation: 2016-02-29 to 2016-03-27
Test: 2016-03-28 to 2016-04-24

Chronological split validation: PASSED


### 2.15 Missing Value Assessment After Feature Engineering

In [16]:

# Features requiring historical observations
historical_features = [
    "lag_28",
    "lag_35",
    "rolling_mean_7",
    "rolling_mean_28"
]

# Check missing values in each split
for name, df in [
    ("Training", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    print(f"\n{name} Missing Values:")
    print(df[historical_features].isna().sum())

# Check missing values across all columns
print("\nAll columns with missing values in training:")
print(
    train_data.isna().sum()[
        train_data.isna().sum() > 0
    ]
)



Training Missing Values:
lag_28              85360
lag_35             106696
rolling_mean_7     103648
rolling_mean_28    167645
dtype: int64

Validation Missing Values:
lag_28             12
lag_35             19
rolling_mean_7     18
rolling_mean_28    39
dtype: int64

Test Missing Values:
lag_28              0
lag_35              0
rolling_mean_7      0
rolling_mean_28    11
dtype: int64

All columns with missing values in training:
event_name_1       4164532
event_type_1       4164532
event_name_2       4522468
event_type_2       4522468
lag_28               85360
lag_35              106696
rolling_mean_7      103648
rolling_mean_28     167645
dtype: int64


### 2.16 Define Initial Modeling Features

In [17]:

# Historical demand features
historical_features = [
    "lag_28",
    "lag_35",
    "rolling_mean_7",
    "rolling_mean_28"
]

# Calendar features
calendar_features = [
    "day_of_week",
    "is_weekend",
    "day_of_month",
    "month",
    "quarter",
    "year",
    "is_event",
    "is_christmas",
    "snap_CA"
]

# Product identifiers
categorical_features = [
    "item_id",
    "dept_id",
    "cat_id"
]

# Target variable
target = "sales"

# Final initial feature list
feature_cols = (
    historical_features
    + calendar_features
    + categorical_features
)

print("Total modeling features:", len(feature_cols))

print("\nSelected features:")
for feature in feature_cols:
    print("-", feature)

print("\nMissing feature columns:",
      set(feature_cols) - set(model_data.columns))


Total modeling features: 16

Selected features:
- lag_28
- lag_35
- rolling_mean_7
- rolling_mean_28
- day_of_week
- is_weekend
- day_of_month
- month
- quarter
- year
- is_event
- is_christmas
- snap_CA
- item_id
- dept_id
- cat_id

Missing feature columns: set()


### 2.17 Prepare Model-Ready Datasets

In [18]:

# Remove training rows without complete historical features
train_ready = train_data.dropna(
    subset=historical_features
).copy()

# Preserve all validation and test observations
val_ready = val_data.copy()
test_ready = test_data.copy()

# Convert categorical features
for df in [train_ready, val_ready, test_ready]:
    for col in categorical_features:
        df[col] = pd.Categorical(
            df[col],
            categories=model_data[col].unique()
        )

# Verify dataset dimensions
print("Training observations:", f"{len(train_ready):,}")
print("Validation observations:", f"{len(val_ready):,}")
print("Test observations:", f"{len(test_ready):,}")

# Check remaining missing historical features
print("\nTraining missing historical values:")
print(train_ready[historical_features].isna().sum())

# Confirm consistent categorical data types
print("\nCategorical feature types:")
print(train_ready[categorical_features].dtypes)


Training observations: 4,364,506
Validation observations: 85,372
Test observations: 85,372

Training missing historical values:
lag_28             0
lag_35             0
rolling_mean_7     0
rolling_mean_28    0
dtype: int64

Categorical feature types:
item_id    category
dept_id    category
cat_id     category
dtype: object


### 2.18 Export Model-Ready Datasets

In [21]:

from pathlib import Path

# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Select required columns without duplicates
export_cols = list(dict.fromkeys([
    "date", "item_id", "store_id",
    target, *feature_cols
]))

# Verify column uniqueness
assert len(export_cols) == len(set(export_cols))

# Export datasets
train_ready[export_cols].to_parquet(
    output_dir / "train.parquet",
    index=False
)

val_ready[export_cols].to_parquet(
    output_dir / "validation.parquet",
    index=False
)

test_ready[export_cols].to_parquet(
    output_dir / "test.parquet",
    index=False
)

print("Processed datasets saved successfully!")

for filename in [
    "train.parquet",
    "validation.parquet",
    "test.parquet"
]:
    path = output_dir / filename
    print(f"{filename}: {path.stat().st_size / 1024**2:.2f} MB")


Processed datasets saved successfully!
train.parquet: 18.28 MB
validation.parquet: 0.32 MB
test.parquet: 0.32 MB



## 3. Final Observations and Conclusion

### 3.1 Data Integration

Three Walmart M5 datasets—historical sales, calendar information,
and weekly selling prices—were integrated successfully.

- Selected store: CA_1
- Unique products: 3,049
- Product categories: FOODS, HOUSEHOLD, and HOBBIES
- Original daily sales observations: 5,832,737
- Final integrated dataset: 5,832,737 rows × 19 columns

The sales data was transformed from wide to long format,
with each observation representing the daily unit sales
of a specific product.

### 3.2 Missing Price Investigation

The initial integration revealed 1,129,842 observations
(19.37%) with missing selling prices.

Further investigation showed that:

- 100% of missing-price observations had zero recorded sales.
- All missing prices occurred before each product's first
  recorded selling price.
- No missing prices were identified between the first and
  last recorded price dates.

These findings suggest that the missing-price observations
are associated with periods before products entered the
recorded assortment.

These observations were excluded from the initial modeling
dataset under a documented pre-introduction assumption.

The resulting dataset contains 4,702,895 observations
with complete selling price information.

Price availability does not independently establish stock
availability, so this assumption remains a limitation.

### 3.3 Time-Series Validation

The filtered dataset was sorted chronologically by product
and date.

Validation confirmed:

- No duplicate product-date records.
- No gaps between consecutive dates within product histories.
- A maximum interval of one day between consecutive records.

These results support the construction of reliable
historical demand features.

### 3.4 Feature Engineering

A total of 16 predictors were selected across three groups.

Historical demand features:
- lag_28
- lag_35
- rolling_mean_7
- rolling_mean_28

Calendar and event features:
- day_of_week
- is_weekend
- day_of_month
- month
- quarter
- year
- is_event
- is_christmas
- snap_CA

Categorical features:
- item_id
- dept_id
- cat_id

Historical demand features were shifted by at least 28 days
to support the planned forecasting horizon.

Selling prices were excluded from the initial feature set
because future selling prices may not be known at the time
of prediction.

### 3.5 Chronological Data Splitting

The dataset was divided chronologically into training,
validation, and testing periods.

Training:
- End date: 2016-02-28
- Final observations: 4,364,506

Validation:
- Period: 2016-02-29 to 2016-03-27
- Observations: 85,372

Testing:
- Period: 2016-03-28 to 2016-04-24
- Observations: 85,372

The validation and testing periods each represent a
28-day forecasting horizon.

Training observations without sufficient historical demand
features were excluded. All validation and testing
observations were retained.

No random splitting was performed.

### 3.6 Final Modeling Dataset

The final datasets contain:

- 4,364,506 training observations
- 85,372 validation observations
- 85,372 testing observations
- 16 selected predictive features
- Daily unit sales as the prediction target

The datasets were exported in Parquet format for efficient
storage and loading during model development.

### 3.7 Limitations and Modeling Considerations

Several considerations remain before training:

1. Missing selling prices were treated as evidence of
   likely pre-introduction periods, not confirmed stockouts.

2. Some validation and testing observations contain
   missing historical features because of insufficient
   product history.

3. The initial modeling scope covers only store CA_1.
   Model performance may not generalize to other stores.

4. Forecast-origin validation must confirm that every
   prediction uses only information available at the
   beginning of its 28-day forecasting window.

5. Forecasting performance should be evaluated across
   categories and products with different demand patterns.

### 3.8 Conclusion

The preprocessing pipeline successfully transformed the
Walmart M5 datasets into structured, product-level
time-series data suitable for machine learning experiments.

The resulting datasets preserve chronological order,
incorporate historical demand and calendar features,
and establish separate training, validation, and testing
periods.

The next phase will focus on establishing forecasting
baselines, validating the 28-day prediction strategy,
training machine learning models, and comparing their
performance.

These forecasting models will subsequently support the
decision-making component of the Agentic AI business
automation system.
